# RAG Step 4: Passing Context to Gemini / Groq LLM

This notebook takes the retrieved context from ChromaDB and sends it to a generative model. The model uses the context to answer the user's question in a grounded way.

Set either `GROQ_API_KEY` or `GEMINI_API_KEY` before running this notebook.


In [ ]:
!pip install -q chromadb sentence-transformers google-generativeai requests python-dotenv

import os
import requests
import chromadb
from chromadb.utils import embedding_functions

# If using Gemini, uncomment and set your API key:
# os.environ['GEMINI_API_KEY'] = 'your_key_here'
# If using Groq, uncomment and set your API key:
# os.environ['GROQ_API_KEY'] = 'your_key_here'

documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a language model to answer questions using external knowledge.",
    "In a typical RAG workflow, a document is chunked, embedded, stored in a vector database, and then retrieved based on semantic similarity.",
    "ChromaDB is a popular vector database that stores embeddings and supports similarity search using metadata and filters.",
    "Large language models can hallucinate when they do not have enough context. RAG reduces this by grounding responses in relevant retrieved passages.",
    "The process usually includes loading documents, splitting them into chunks, embedding the chunks, storing them, and then querying the database."
]

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.get_or_create_collection(name='rag_demo_collection', embedding_function=embedding_fn)

if collection.count() == 0:
    collection.add(
        documents=documents,
        metadatas=[{'source': f'doc_{i}'} for i in range(1, len(documents) + 1)],
        ids=[f'chunk_{i}' for i in range(1, len(documents) + 1)]
    )

query = 'How does RAG reduce hallucinations in LLM outputs?'
retrieved = collection.query(
    query_texts=[query],
    n_results=3,
    include=['documents']
)
context = '\n\n'.join(retrieved['documents'][0])

def ask_llm(question, context):
    if os.getenv('GROQ_API_KEY'):
        url = 'https://api.groq.com/openai/v1/chat/completions'
        headers = {
            'Authorization': f"Bearer {os.environ['GROQ_API_KEY']}",
            'Content-Type': 'application/json'
        }
        payload = {
            'model': 'llama-3.1-8b-instant',
            'messages': [
                {'role': 'system', 'content': 'You answer using ONLY the provided context. If the answer is not in the context, say so clearly.'},
                {'role': 'user', 'content': f"Context:\n{context}\n\nQuestion:\n{question}"}
            ],
            'temperature': 0.2
        }
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']

    if os.getenv('GEMINI_API_KEY'):
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        model = genai.GenerativeModel('gemini-1.5-flash')
        prompt = f"Use only the provided context.\n\nContext:\n{context}\n\nQuestion:\n{question}"
        response = model.generate_content(prompt)
        return response.text

    return 'No API key found. Set GROQ_API_KEY or GEMINI_API_KEY and rerun this notebook.'

answer = ask_llm(query, context)
print('Query:', query)
print('--- Retrieved context ---')
print(context)
print('--- Final answer ---')
print(answer)


## Final idea

In a real RAG pipeline, the retriever finds the right passages, and the LLM uses those passages to craft a grounded answer. This is the core concept behind reliable question-answering systems.
